In [17]:
import numpy as np
# --- Execution Example ---
query = np.random.rand(512)
docs = np.random.rand(30000, 512)



In [18]:
def mmr_rerank(query_vector, doc_embeddings, top_k=4, lmbda=0.5):
    """
    Args:
        query_vector: np.array of shape (512,)
        doc_embeddings: np.array of shape (30, 512)
        top_k: Number of documents to return
        lmbda: Diversity balance factor (0.5)
    """
    # 1. Ensure vectors are normalized for pure cosine similarity via dot product
    q_norm = query_vector / np.linalg.norm(query_vector)
    d_norms = doc_embeddings / np.linalg.norm(doc_embeddings, axis=1, keepdims=True)
    
    # 2. Precompute similarity matrix between all docs and the query (30x1)
    query_similarities = np.dot(d_norms, q_norm)
    
    # 3. Precompute similarity matrix between all docs (30x30)
    # This makes checking redundancy against selected docs incredibly fast
    doc_similarities = np.dot(d_norms, d_norms.T)
    
    selected_indices = []
    remaining_indices = list(range(len(doc_embeddings)))
    
    # First selection is purely based on the highest query similarity
    first_choice = np.argmax(query_similarities)
    selected_indices.append(first_choice)
    remaining_indices.remove(first_choice)
    
    # Iteratively select the remaining top_k - 1 documents
    while len(selected_indices) < top_k and remaining_indices:
        best_mmr_score = -float('inf')
        best_candidate = -1
        
        for idx in remaining_indices:
            relevance = query_similarities[idx]
            
            # Find max similarity between this candidate and all already selected docs
            redundancy = max(doc_similarities[idx, sel_idx] for sel_idx in selected_indices)
            
            # MMR Equation
            mmr_score = lmbda * relevance - (1 - lmbda) * redundancy
            
            if mmr_score > best_mmr_score:
                best_mmr_score = mmr_score
                best_candidate = idx
                
        selected_indices.append(best_candidate)
        remaining_indices.remove(best_candidate)
        
    return selected_indices



In [19]:
import numpy as np

def mmr_rerank_vectorized(query_vector, doc_embeddings, top_k=4, lmbda=0.5):
    """
    Completely vectorized MMR implementation without per-candidate loops.
    
    Args:
        query_vector: np.array of shape (512,)
        doc_embeddings: np.array of shape (N, 512)
        top_k: Number of documents to return
        lmbda: Diversity balance factor (0 to 1)
    """
    num_docs = doc_embeddings.shape[0]
    top_k = min(top_k, num_docs)
    
    # 1. Normalize vectors for unit-length dot products (Cosine Similarity)
    q_norm = query_vector / np.linalg.norm(query_vector)
    d_norms = doc_embeddings / np.linalg.norm(doc_embeddings, axis=1, keepdims=True)
    
    # 2. Precompute similarities
    query_sims = np.dot(d_norms, q_norm)  # Shape: (N,)
    doc_sims = np.dot(d_norms, d_norms.T)  # Shape: (N, N)
    
    # 3. Track selected state and running maximum redundancy
    # Mask to prevent re-selecting already chosen documents
    selected_mask = np.zeros(num_docs, dtype=bool)
    # Stores the max similarity between any doc and the already selected set
    max_redundancy = np.zeros(num_docs)
    
    selected_indices = []
    
    # --- Step 1: Pure relevance selection ---
    first_choice = np.argmax(query_sims)
    selected_indices.append(first_choice)
    selected_mask[first_choice] = True
    
    # --- Step 2 to K: Iterative Vectorized Matrix Selection ---
    for _ in range(1, top_k):
        # Update the redundancy array for all items relative to the latest pick
        latest_pick = selected_indices[-1]
        
        # Element-wise maximum across the new similarity column and prior history
        max_redundancy = np.maximum(max_redundancy, doc_sims[:, latest_pick])
        
        # Calculate MMR scores for all N elements simultaneously
        mmr_scores = lmbda * query_sims - (1 - lmbda) * max_redundancy
        
        # Force already selected elements to -infinity so they are ignored
        mmr_scores[selected_mask] = -np.inf
        
        # Find the single best candidate index across the vector
        next_choice = np.argmax(mmr_scores)
        
        selected_indices.append(next_choice)
        selected_mask[next_choice] = True
        
    return selected_indices


In [20]:
import time

In [21]:
init = time.perf_counter_ns()
final_four_docs = mmr_rerank(query, docs, top_k=4, lmbda=0.5)
print(f"MMR rerank execution time: {(time.perf_counter_ns() - init) / 1e6:.2f} ms")
print("Selected document indices:", final_four_docs)

MMR rerank execution time: 10826.55 ms
Selected document indices: [np.int64(7090), 11046, 16618, 6584]


In [22]:
init = time.perf_counter_ns()
final_four_docs_vectorized = mmr_rerank_vectorized(query, docs, top_k=4, lmbda=0.5)
print(f"MMR rerank vectorized execution time: {(time.perf_counter_ns() - init) / 1e6:.2f} ms")
print("Selected document indices (vectorized):", final_four_docs_vectorized)

MMR rerank vectorized execution time: 10011.30 ms
Selected document indices (vectorized): [np.int64(7090), np.int64(11046), np.int64(16618), np.int64(6584)]


In [23]:

def mmr_fetch_then_rerank(query_vector, doc_embeddings, top_k=4, fetch_k=50, lmbda=0.5):
    """
    Production-ready MMR. Pre-filters to fetch_k candidates 
    before performing local vectorization.
    
    Args:
        query_vector: np.array of shape (512,)
        doc_embeddings: np.array of shape (N, 512)
        top_k: Number of diverse documents to output (e.g., 4)
        fetch_k: Size of the preliminary candidate pool (e.g., 50)
        lmbda: Balance factor (0.5)
    """
    num_docs = doc_embeddings.shape[0]
    fetch_k = min(fetch_k, num_docs)
    top_k = min(top_k, fetch_k)
    
    # 1. Unit normalization for clean dot-product cosine similarity
    q_norm = query_vector / np.linalg.norm(query_vector)
    d_norms = doc_embeddings / np.linalg.norm(doc_embeddings, axis=1, keepdims=True)
    
    # 2. Stage 1: Get raw similarity against all documents (1xN dot product)
    global_query_sims = np.dot(d_norms, q_norm)
    
    # 3. Extract the top_k relevant candidate indices using argpartition.
    # This is an O(N) operation—significantly faster than full sorting.
    candidate_indices = np.argpartition(global_query_sims, -fetch_k)[-fetch_k:]
    
    # Sort candidates in descending order of relevance 
    candidate_indices = candidate_indices[np.argsort(-global_query_sims[candidate_indices])]
    
    # 4. Stage 2: Create a local pool for the MMR calculations
    local_embeddings = d_norms[candidate_indices]
    local_query_sims = global_query_sims[candidate_indices]
    
    # Precompute a small, fast local similarity matrix (e.g., 50x50)
    local_doc_sims = np.dot(local_embeddings, local_embeddings.T)
    
    # 5. Execute vectorized MMR tracking loops on the local pool
    selected_mask = np.zeros(fetch_k, dtype=bool)
    max_redundancy = np.zeros(fetch_k)
    local_selected_positions = []
    
    # The first selection is always index 0 (the absolute highest similarity)
    first_choice = 0
    local_selected_positions.append(first_choice)
    selected_mask[first_choice] = True
    
    for _ in range(1, top_k):
        latest_pick = local_selected_positions[-1]
        
        # Track running maximum redundancy using the 50x50 slice
        max_redundancy = np.maximum(max_redundancy, local_doc_sims[:, latest_pick])
        
        # Standard MMR vector calculation
        mmr_scores = lmbda * local_query_sims - (1 - lmbda) * max_redundancy
        mmr_scores[selected_mask] = -np.inf
        
        next_choice = np.argmax(mmr_scores)
        local_selected_positions.append(next_choice)
        selected_mask[next_choice] = True
        
    # 6. Map the local pool selections back to your absolute dataset indices
    return [candidate_indices[pos] for pos in local_selected_positions]


In [24]:
init = time.perf_counter_ns()
final_four_docs_fetch_then_rerank = mmr_fetch_then_rerank(query, docs, top_k=4, fetch_k=50, lmbda=0.5)
print(f"MMR fetch-then-rerank execution time: {(time.perf_counter_ns() - init) / 1e6:.2f} ms")
print("Selected document indices (fetch-then-rerank):", final_four_docs_fetch_then_rerank)

MMR fetch-then-rerank execution time: 272.61 ms
Selected document indices (fetch-then-rerank): [np.int64(7090), np.int64(2599), np.int64(1582), np.int64(23626)]
